In [1]:

import json
import pennylane as qml
import pennylane.numpy as np
import scipy

H = np.array([[ 0.39488016,  0.04722628, -0.17943126, -0.03673282],
        [ 0.04722628,  0.37758558,  0.12997088,  0.17848188],
        [-0.17943126,  0.12997088,  0.53582574,  0.01115543],
        [-0.03673282,  0.17848188,  0.01115543,  0.29170851]])

U = scipy.linalg.expm(2j * np.pi * H)
U_powers = [scipy.linalg.fractional_matrix_power(U, 2**i) for i in range(6)]

def state_prep(params, wires):
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[2], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])

dev = qml.device('default.qubit', wires = range(8))

@qml.qnode(dev)
def qpe_circuit(params):
    state_prep(params, wires=[6,7])
    for i in range(6):
        qml.Hadamard(wires=i)
    for i in range(6):
        qml.ControlledQubitUnitary(U_powers[i], control_wires=i, wires=[6,7])
    qml.adjoint(qml.QFT)(wires=range(6))
    return qml.probs(wires=range(6))

def compute_statistics(params):
    probs = qpe_circuit(params)
    phases = np.arange(2**6)/2**6
    mu = np.sum(probs*phases)
    sigma = np.sqrt(np.sum(probs*(phases-mu)**2))
    return mu.item(), sigma.item()

def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    outs = list(compute_statistics(ins))
    return json.dumps(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, atol = 1e-4)

test_cases = [
    ('[1.35889209, -0.6219561, -1.31577162]', '[0.121662, 0.120539]'),
    ('[4.40084815, -0.77288063,  0.6425846]', '[0.303264, 0.043692]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")
    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")
    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


Running test case 0 with input '[1.35889209, -0.6219561, -1.31577162]'...
Wrong Answer. Have: '[0.4225101124540759, 0.28793368521821644]'. Want: '[0.121662, 0.120539]'.
Running test case 1 with input '[4.40084815, -0.77288063,  0.6425846]'...
Wrong Answer. Have: '[0.4550617816406146, 0.35601616398337393]'. Want: '[0.303264, 0.043692]'.
